In [1]:
# imports
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')
from solver.quantum_solver.dwave_simulator import DwaveSimulator

sim = DwaveSimulator()
print('Import OK')

ModuleNotFoundError: No module named 'solver'

In [ ]:
# génération d'une instance Ising aléatoire
def random_ising(n, seed=42):
    rng = np.random.default_rng(seed)
    h = {i: rng.uniform(-1, 1) for i in range(n)}
    J = {(i, j): rng.uniform(-1, 1)
         for i in range(n) for j in range(i + 1, n)}
    return h, J

N = 6
h, J = random_ising(N)
print(f'n={N}')
print(f'h = {h}')
print(f'J = {J}')

In [ ]:
# simulation et tracé des 5 plus petites valeurs propres
NB_EIGEN = 5
eigenvalues = sim.simulate_evolution(h, J, nb_eigenvalues=NB_EIGEN)
# eigenvalues : tableau (101, NB_EIGEN)

steps = np.arange(101)

plt.figure(figsize=(9, 5))
for k in range(NB_EIGEN):
    plt.plot(steps, eigenvalues[:, k], marker='o', markersize=2, label=f'Eigen {k+1}')
plt.xlabel('Fraction d\'annealing')
plt.ylabel('Energie')
plt.title(f'5 valeurs propres les plus basses (n={N})')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# calcul et tracé du spectral gap
spectral_gap = eigenvalues[:, 1] - eigenvalues[:, 0]
g_min = np.min(spectral_gap)
g_min_idx = np.argmin(spectral_gap)

plt.figure(figsize=(9, 4))
plt.plot(steps, spectral_gap, marker='o', markersize=2,
         label='gap ε₁ - ε₀', color='steelblue')
plt.axvline(g_min_idx, color='red', linestyle='--', alpha=0.6,
            label=f'g_min = {g_min:.4f} à l\'étape {g_min_idx}')
plt.xlabel('Fraction d\'annealing')
plt.ylabel('Gap d\'énergie')
plt.title('Spectral gap pendant l\'annealing')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Gap spectral minimum : {g_min:.6f} à l\'étape {g_min_idx}')

In [ ]:
# vérification que le ground state correspond bien à la solution optimale
gs_energy, gs_vec = sim.get_ground_state(h, J)

# l'indice de plus grande amplitude donne la configuration de spins
best_idx = np.argmax(np.abs(gs_vec) ** 2)
spin_config = [1 if (best_idx >> (N - 1 - k)) & 1 == 0 else -1 for k in range(N)]

ising_energy = (sum(h[i] * spin_config[i] for i in range(N))
              + sum(J[(i,j)] * spin_config[i] * spin_config[j] for (i,j) in J))

print(f'Energie ground state (H_final)          : {gs_energy:.6f}')
print(f'Plus petite valeur propre (dernier pas)  : {eigenvalues[-1, 0]:.6f}')
print(f'Configuration de spins la plus probable  : {spin_config}')
print(f'Energie Ising de cette configuration     : {ising_energy:.6f}')

In [ ]:
# évolution du gap spectral selon la taille du problème (3 à 10 variables)
sizes = range(3, 11)
min_gaps = []

for n in sizes:
    h_n, J_n = random_ising(n, seed=0)
    eigs = sim.simulate_evolution(h_n, J_n, nb_eigenvalues=2)
    gap = np.min(eigs[:, 1] - eigs[:, 0])
    min_gaps.append(gap)
    print(f'n={n:2d} -> gap min = {gap:.6f}')

plt.figure(figsize=(8, 4))
plt.plot(list(sizes), min_gaps, 'o-')
plt.xlabel('Nombre de qubits n')
plt.ylabel('Gap spectral minimum')
plt.title('Gap spectral vs taille du problème')
plt.tight_layout()
plt.show()

# le gap tend à décroître exponentiellement avec n
# => temps d'annealing nécessaire ~1/g²_min, donc croît exponentiellement

In [ ]:
# effet du rescaling des poids de H_final sur le gap
n_test = 5
h_base, J_base = random_ising(n_test, seed=7)
scale_factors = [0.1, 0.25, 0.5, 1.0, 2.0, 5.0, 10.0]
scaled_gaps = []

for scale in scale_factors:
    h_sc = {i: v * scale for i, v in h_base.items()}
    J_sc = {k: v * scale for k, v in J_base.items()}
    eigs = sim.simulate_evolution(h_sc, J_sc, nb_eigenvalues=2)
    gap = np.min(eigs[:, 1] - eigs[:, 0])
    scaled_gaps.append(gap)
    print(f'scale={scale:5.2f} -> gap min = {gap:.6f}')

plt.figure(figsize=(8, 4))
plt.semilogx(scale_factors, scaled_gaps, 'o-')
plt.xlabel('Facteur d\'échelle')
plt.ylabel('Gap spectral minimum')
plt.title('Gap spectral vs rescaling de H_final')
plt.tight_layout()
plt.show()

# réduire les poids (scale < 1) pour rentrer dans la plage D-Wave
# diminue le gap absolu => rapport signal/bruit dégradé => résolution moins bonne